In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import tensorflow as tf
from math import ceil
import cv2
import matplotlib.pyplot as plt
from albumentations import RandomCrop, Compose, HorizontalFlip, VerticalFlip, OneOf
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, InputLayer, GlobalAveragePooling2D, Input, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model

import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.applications import ResNet50, InceptionResNetV2
from tensorflow.keras.layers import Dense, Input, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import Model
from tensorflow.keras.optimizers import Adam

2026-04-28 16:38:18.364188: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-28 16:38:18.374392: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777394298.385476      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777394298.388964      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 16:38:18.401971: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
# def create_model(input_shape):
#     input = Input(shape = input_shape)
    
#     #Create and complite model and show sumarry
    
#     x_model = ResNet50(weights = None, include_top = False, input_tensor = input, pooling = None,
#                             classes = None)
#     for layer in x_model.layers:
#         layer.trainable = True
        
#     x = GlobalAveragePooling2D()(x_model.output)
#     x = BatchNormalization()(x)
    
#     x = Dense(1024, activation = "relu")(x)
#     x = BatchNormalization()(x)
#     x = Dense(512, activation = "relu")(x)
    
#     healthy = Dense(5, activation = "softmax", name="output_layer")(x)
    
#     model = Model(inputs = x_model.input, outputs = healthy)
    
#     return model 

In [4]:
gm_exp = tf.Variable(3.0, dtype = tf.float32)
def generalized_mean_pool_2d(X):
    pool = (tf.reduce_mean(tf.abs(X**(gm_exp)),
                                axis = [1, 2],
                                keepdims = False) + 1.e-7)**(1./gm_exp)
    return pool


def create_model(input_shape):
    input = Input(shape = input_shape)

    #Create and complite model and show summary

    x_model = ResNet50(weights = None , include_top = False, input_tensor = input, pooling = None,
                        classes = None)
    for layer in x_model.layers:
        layer.trainable = True

    # Gem
    lambda_layer = Lambda(generalized_mean_pool_2d)
    lambda_layer.trainable_weights.extend([gm_exp])
    x = lambda_layer(x_model.output)

    #output
    # x = Dense(1024, activation = "relu")(x)
    # x = Dense(512, activation = "relu")(x)

    #output
    healthy = Dense(5, activation = 'softmax', name = 'plan_diseases')(x)


    #model
    model = Model(inputs = x_model.input, outputs = healthy )

    return model

2026-04-28 16:38:26.622857: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
path = "/kaggle/input/cassava-leaf-disease-classification/"
img_dir = path + "test_images/"
sample_submission = pd.read_csv("../input/cassava-leaf-disease-classification/sample_submission.csv")
# train = pd.read_csv("../input/cassava-leaf-disease-classification/train.csv")
# sample_submission = pd.DataFrame()
# sample_submission['image_id'] = train['image_id']

In [6]:
### HELPER FUNCTIONS

def pred_checker(m, arr):
    for index, i in enumerate(arr):
        if i == m:
            return index + 1
    
def max_in_numpy(arr):
    m = arr[0]
    for i in arr:
        if m < i:
            m = i
    return m

In [7]:
### TEST_DATA_GENERATOR
# augmentation should be function already, ec HorizontalFlip()
def _read(path):
    img = cv2.imread(path)    
    return img

class TestDataGenerator(tf.keras.utils.Sequence):
    
# img_dir deleted, restore if needed.    
    
    def __init__(self , X_set, ids, augmentation = None, img_dir = "../input/cassava-leaf-disease-classification/test_images/",
                                                            batch_size = 8, img_size = (224,224,3), do_aug = False):
        self.X = X_set
        self.batch_size = batch_size
        self.ids = ids
        self.img_size  = img_size  
        self.img_dir = img_dir
        self.on_epoch_end()
        self.augmentation = augmentation
        self.do_aug = do_aug
        
            
    def __len__(self):
        return int(ceil(len(self.ids)/self.batch_size))

    def __getitem__(self, index):
        indices = self.ids[index*self.batch_size:(index+1)*self.batch_size]
        X = self.__generator__(indices)
        return X
    
    def on_epoch_end(self):
        self.indices = np.arange(len(self.ids))
        

    def __generator__(self, indices):
        X = np.empty((self.batch_size, *self.img_size))
        for i, index in enumerate(indices):
            ID = self.X['image_id'][index]
            image = cv2.imread(self.img_dir + self.X['image_id'][i])
            image = cv2.resize(image, (224,224))
#             if self.do_aug:
#                 for augments in self.augmentation:
#                     image = augments(image=image)['image']
            X[i,] = image/255
        return X
        

In [8]:
sample_submission

,image_id,label
0,1234294272.jpg,4
1,1234332763.jpg,4
2,1234375577.jpg,4
3,1234555380.jpg,4
4,1234571117.jpg,4
...,...,...
2671,2656351084.jpg,4
2672,2656378111.jpg,4
2673,2656781269.jpg,4
2674,2656964450.jpg,4


In [9]:
# ids_train = train.index
# X_set = sample_submission
# Y_set = pd.get_dummies(train.label)
# ids_train = np.array(X_set.index)
# for X,y in TestDataGenerator(X_set, img_dir, ids_test):
#     break
    

# print(X.shape)
# print(y)

In [10]:
# prediction = []
# prediction_horizontal = []
# prediction_vertical = []
# prediction_h_v = []
# names = ["0_Cassava_Leaf_Disease_solo_seresnet50 (1).h5", "1_Cassava_Leaf_Disease_solo_seresnet50 (1).h5", "2_Cassava_Leaf_Disease_solo_seresnet50 (1).h5"]
# test_set = sample_submission
# del test_set["label"]
# ids = sample_submission.index
# for i in names:
#     model = create_model((224, 224, 3))
#     model.load_weights("../input/weights/" + i)
#     data_generator = TestDataGenerator(X_set = test_set, ids = ids, batch_size = 1)
    
# #     data_generator_horizontal = TestDataGenerator(X_set = test_set, ids = ids, batch_size = 1,
# #                                                   augmentation = [HorizontalFlip()], do_aug = True)
    
# #     data_generator_vertical = TestDataGenerator(X_set = test_set, ids = ids, batch_size = 1,
# #                                                 augmentation = [VerticalFlip()], do_aug = True)
    
# #     data_generator_v_h = TestDataGenerator(X_set = test_set, ids = ids, batch_size = 1,
# #                                           augmentation = [HorizontalFlip(), VerticalFlip()])
    
    
    
    
#     pred = model.predict_generator(data_generator, verbose = 1)
# #     pred_h = model.predict_generator(data_generator_horizontal, verbose = 1)
# #     pred_v = model.predict_generator(data_generator_vertical, verbose = 1)
# #     pred_h_v = model.predict_generator(data_generator_v_h, verbose = 1)
# #    all_current_preds = (pred)
#     prediction.append(pred)

In [11]:
X_set = sample_submission
del X_set['label']
path_weights = "../input/weights/"
models = os.listdir(path_weights)
ids_test = sample_submission.index
data_generator_test = TestDataGenerator(X_set, ids_test, img_dir=img_dir)
all_pred = []
input_shape = (224, 224, 3) ### I FORGOT THE FUCKIN SIZE LOL

for w in models:
    model = create_model(input_shape)
    model.load_weights(path_weights + w)
    preds = model.predict_generator(data_generator_test, verbose = 1)
    preds = preds[:sample_submission.shape[0]]
    all_pred.append(preds)
    
sum_pred = 0
for p in all_pred:
    sum_pred += p
sum_pred = sum_pred/len(all_pred)

diagnos = []
for pr in sum_pred:
    diagnos.append(np.argmax(np.array(pr)))  
    
sample_submission.label = diagnos

FileNotFoundError: [Errno 2] No such file or directory: '../input/weights/'

In [12]:
# from sklearn.metrics import accuracy_score
# predicted = sample_submission.label
# truth = train.head(100).label
# accuracy_score(truth, predicted)

In [13]:
sample_submission['label'] = diagnos

NameError: name 'diagnos' is not defined

In [14]:
sample_submission.to_csv('submission.csv', index = False)

In [15]:
# prediction = []
# ids = sample_submission.index
# test_set = sample_submission
# data_generator_test = TestDataGenerator(X_set = test_set, ids = ids, batch_size = 1)
# pred = model.predict_generator(data_generator_test, verbose = 1)
# for i in pred:
#     m = max_in_numpy(i)
#     ind = pred_checker(m, i)
#     prediction.append(ind)
# sample_submission['label'] = prediction
# sample_submission

In [16]:
# img = cv2.imread("../kaggle/input/cassava-leaf-disease-classification/test_images/2216849948.jpg")
# img2 = cv2.imread("../input/cassava-leaf-disease-classification/test_images/2216849948.jpg")
# print(img)
# print(img2)

In [17]:
# normal_image = cv2.imread("../input/cassava-leaf-disease-classification/test_images/2216849948.jpg")
# plt.imshow(normal_image)

In [18]:
# a = [ VerticalFlip()]
# image = cv2.imread("../input/cassava-leaf-disease-classification/test_images/2216849948.jpg")
# for i in a:
#     image = i(image=image)["image"]
# plt.imshow(image)